# PCB Defect Synthesis & Detection Pipeline (Kaggle Edition)

**Paper tham chiếu**: *A Synthetic Defect Image Generation Method for Enhanced Industrial Defect Detection Based on Inpainting Diffusion and Context-aware Mask Generation* (Journal of Intelligent Manufacturing, 2026).

Repo này là bản **resource-limited**, không tái lập đầy đủ 5 mô hình của paper:
- Chỉ benchmark **YOLO11s**.
- Dataset local có 7 seen boards và 3 unseen boards (10/11/12).
- Step 2 dùng fp16 + gradient checkpointing; batch vật lý = 1, `--lora_batch` là gradient accumulation.
- Pipeline dùng **một GPU**; không tự động DataParallel qua nhiều GPU.

## Trước khi bấm Save & Run All

1. Trên máy local, chạy `python package_for_kaggle.py` để tạo `diffusion_kaggle_code.zip`.
2. Upload file zip thành **Kaggle Dataset** tên `diffusion-code`.
3. Upload `data/pcb-defect-dataset/` thành **Kaggle Dataset** tên `pcb-defect-dataset`.
4. Trong notebook: **Add Input** cả hai dataset.
5. **Settings**: Accelerator = GPU (T4/P100), **Internet = On**.
6. **Save Version -> Save & Run All (Commit) -> Save**.

Sau khi Save, có thể đóng trình duyệt và tắt máy; Kaggle chạy trên máy chủ.

> Nếu không add code zip, notebook fallback clone GitHub. Bản GitHub có thể chưa có fix mới trong repo local, nên **ưu tiên code zip**.

## Profile của notebook này

| Tham số | Giá trị |
|---|---:|
| CAMG epochs | 10 |
| LoRA epochs | 10 |
| Synthetic samples | 400 |
| YOLO judge epochs | 100 |
| Benchmark epochs | 100 |

Đây là profile để cố gắng hoàn tất trong một commit Kaggle, **không phải full paper**.
`--resume` dùng lại artifact hợp lệ; Step 2 lưu checkpoint sau mỗi epoch để có thể resume nếu commit bị timeout.


## Bước 1: Kiểm Tra Cấu Hình GPU

Pipeline chỉ dùng một GPU. Nếu Kaggle cấp T4 x2, GPU còn lại không được dùng để DataParallel.


In [ ]:
# ── 1. Kiểm tra cấu hình GPU ─────────────────────────────
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"GPU Device Count: {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {props.name} - {props.total_memory / (1024**3):.1f} GB VRAM")

if device_count >= 1:
    print("\n=> GPU đã sẵn sàng. Pipeline sẽ dùng GPU 0.")
else:
    raise RuntimeError(
        "Chưa bật GPU. Vào Settings -> Accelerator -> GPU T4/P100 "
        "trước khi Save & Run All."
    )


## Bước 2: Chuẩn Bị Mã Nguồn & Cài Đặt Thư Viện Phụ Thuộc

In [ ]:
# ── 2. Chuẩn bị mã nguồn: ưu tiên code zip hiện tại ────────
import os
import shutil
import zipfile
from pathlib import Path

working_dir = Path("/kaggle/working")
repo_dir = working_dir / "diffusion"
code_zip_name = "diffusion_kaggle_code.zip"


def find_code_zip():
    direct = Path("/kaggle/input/diffusion-code") / code_zip_name
    if direct.is_file():
        return direct
    matches = sorted(Path("/kaggle/input").glob(f"**/{code_zip_name}"))
    return matches[0] if matches else None


code_zip = find_code_zip()
if code_zip is not None:
    print(f"Đang giải nén code hiện tại: {code_zip}")
    repo_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall(repo_dir)
elif (repo_dir / "run_pipeline.py").exists():
    print("Dùng code đã có trong /kaggle/working/diffusion.")
elif (working_dir / "run_pipeline.py").exists():
    repo_dir = working_dir
else:
    # Trường hợp Kaggle tự giải nén code zip thành thư mục Input.
    code_dirs = sorted({p.parent for p in Path("/kaggle/input").glob("**/run_pipeline.py")})
    if code_dirs:
        print(f"Copy code từ Kaggle Input: {code_dirs[0]}")
        shutil.copytree(code_dirs[0], repo_dir, dirs_exist_ok=True)
    else:
        print("Không thấy code zip; fallback clone GitHub (có thể thiếu fix mới nhất).")
        !git clone https://github.com/amebett1/diffusion.git /kaggle/working/diffusion

if not (repo_dir / "run_pipeline.py").exists():
    raise RuntimeError(
        "Không tìm thấy run_pipeline.py. Hãy upload diffusion_kaggle_code.zip "
        "thành Kaggle Dataset tên 'diffusion-code' và Add Input."
    )

os.chdir(repo_dir)
print(f"Đang làm việc tại: {os.getcwd()}")

# Cài dependency đúng theo requirements.txt (LoRA native, không cần peft).
%pip install -q -r requirements.txt gdown


## Bước 3: Chuẩn Bị Dataset DeepPCB

Notebook hỗ trợ 2 cách:
- **Cách 1 (khuyến nghị cho commit)**: Add Kaggle Dataset `pcb-defect-dataset` vào Input.
- **Cách 2**: Tải zip từ Google Drive bằng `gdown`.

Cell bên dưới bỏ qua `diffusion_kaggle_code.zip` (đó là code, không phải dataset PCB). Nếu không tìm thấy ảnh PCB trong Kaggle Input, notebook mới fallback Google Drive.


In [ ]:
# ── 3. Chuẩn bị dataset PCB ────────────────────────────────
import os
import shutil
import zipfile
from pathlib import Path

raw_dir = Path("data/pcb-defect-dataset")
raw_dir.mkdir(parents=True, exist_ok=True)

# TÙY CHỌN: điền đường dẫn dataset cụ thể, ví dụ "/kaggle/input/pcb-defect-dataset".
# Để trống để notebook tự quét /kaggle/input.
CUSTOM_KAGGLE_INPUT = ""

kaggle_input = Path("/kaggle/input")
found_in_kaggle = False


def link_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        os.symlink(str(src.resolve()), str(dst.resolve()))
    except Exception:
        if src.is_dir():
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        else:
            shutil.copy2(str(src), str(dst))


def copy_dataset_dir(src_dir: Path) -> None:
    for item in src_dir.iterdir():
        link_or_copy(item, raw_dir / item.name)


# 1. Dataset được chỉ định trực tiếp
if CUSTOM_KAGGLE_INPUT and Path(CUSTOM_KAGGLE_INPUT).exists():
    src_path = Path(CUSTOM_KAGGLE_INPUT)
    print(f"Đang dùng dataset chỉ định tại: {src_path}")
    copy_dataset_dir(src_path)
    found_in_kaggle = True

# 2. Tự quét Kaggle Input
elif kaggle_input.exists():
    # 2a. Giải nén zip dataset PCB; bỏ qua code zip.
    for zip_p in kaggle_input.glob("**/*.zip"):
        if "diffusion_kaggle_code" in zip_p.name.lower():
            continue
        try:
            with zipfile.ZipFile(zip_p, "r") as z:
                names = z.namelist()
                has_image = any(
                    n.lower().endswith((".jpg", ".jpeg", ".png")) for n in names
                )
                if not has_image:
                    continue
                has_root = any(n.startswith("pcb-defect-dataset/") for n in names)
                dest = Path("data") if has_root else raw_dir
                print(f"Đang giải nén dataset PCB: {zip_p.name}")
                z.extractall(dest)
        except zipfile.BadZipFile:
            print(f"Bỏ qua file zip lỗi: {zip_p}")
            continue

        if any(raw_dir.glob("**/*.jpg")):
            found_in_kaggle = True
            print(f"Đã giải nén dataset vào '{raw_dir}'.")
            break

    # 2b. Dataset đã được giải nén sẵn thành thư mục ảnh.
    if not found_in_kaggle and any(kaggle_input.glob("**/*.jpg")):
        sample_jpg = next(kaggle_input.glob("**/*.jpg"))
        candidate = sample_jpg.parent
        while candidate != kaggle_input and candidate.parent != kaggle_input:
            if any((candidate / s).exists() for s in ["train", "val", "test", "images"]):
                break
            candidate = candidate.parent
        print(f"Tự động phát hiện dataset trong Kaggle Input: {candidate}")
        copy_dataset_dir(candidate)
        found_in_kaggle = True

# 3. Fallback Google Drive nếu Kaggle Input không có ảnh PCB.
DATASET_DRIVE_LINK = "https://drive.google.com/file/d/1jpg-Igk_2i8oM6B7N2_mPv6tM7UQJ34y/view?usp=sharing"

if not any(raw_dir.glob("**/*.jpg")):
    print("Không tìm thấy PCB trong Kaggle Input. Đang tải từ Google Drive (khoảng 1.1 GB)...")
    !gdown --fuzzy "{DATASET_DRIVE_LINK}" -O dataset.zip
    with zipfile.ZipFile("dataset.zip", "r") as z:
        names = z.namelist()
        has_root = any(n.startswith("pcb-defect-dataset/") for n in names)
        dest = Path("data") if has_root else raw_dir
        z.extractall(dest)
    for s in ["train", "val", "test", "images"]:
        if (Path("data") / s).exists():
            shutil.move(str(Path("data") / s), str(raw_dir / s))
    print("Đã tải và giải nén dataset từ Google Drive.")

total_imgs = len(list(raw_dir.glob("**/*.jpg")))
print(f"Dữ liệu PCB đã sẵn sàng! Tìm thấy {total_imgs} ảnh trong '{raw_dir}'.")
if total_imgs == 0:
    raise RuntimeError(
        "Không có ảnh PCB. Hãy Add Input dataset 'pcb-defect-dataset' "
        "hoặc kiểm tra lại đường dẫn."
    )


## Bước 4: Tiền Xử Lý Dữ Liệu & Tạo 799 Ảnh Phôi Sạch (Telea Inpainting)
Quy trình tự động thực hiện:
1. Phân chia tập dữ liệu **In-Distribution (ID 10% Low-Data)** và **New Product Introduction (NPI)** theo thiết lập thực nghiệm của bài báo.
2. Áp dụng thuật toán Telea Inpainting để xóa sạch khuyết tật thực tế, sinh chính xác **799 ảnh phôi sạch không lỗi** (`data/pcb_normal/`) theo đúng trang 6 cột 1 của bài báo.
3. Xuất file cấu hình dataset `data/id_setup.yaml` và `data/train_metadata.json`.

In [ ]:
# ── 4. Tiền xử lý dữ liệu và tạo 799 ảnh normal ──────────────
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "run_pipeline.py",
        "prepare_data",
        "--raw_dir",
        "data/pcb-defect-dataset",
    ],
    check=True,
)

# Kiểm tra bảng trạng thái pipeline
subprocess.run(
    [sys.executable, "run_pipeline.py", "status"],
    check=True,
)


---
# PHƯƠNG ÁN A: CHẠY TOÀN BỘ TRONG NỀN (RECOMMENDED)

### Dành cho Kaggle: **Save Version -> Save & Run All (Commit)**

> **Hướng dẫn tắt máy:**
> 1. Bấm **Save Version** ở góc trên bên phải.
> 2. Chọn **Save & Run All (Commit)**.
> 3. Bấm **Save**.
> 4. Có thể đóng trình duyệt và tắt máy.
>
> Máy chủ Kaggle sẽ chạy lệnh bên dưới trong **nhiều giờ** (tùy GPU và số epoch). Cờ `--resume` dùng lại artifact hợp lệ nếu bị gián đoạn; không dùng smoke checkpoint.


In [ ]:
# ── ONE-CLICK FULL PIPELINE RUN (KAGGLE COMMIT PROFILE) ─────
# Tải trước Stable Diffusion Inpainting fp16 để lỗi mạng xuất hiện sớm,
# không phải đợi đến sau khi CAMG/mask đã chạy xong.
import subprocess
import sys

from huggingface_hub import snapshot_download

try:
    snapshot_download(
        repo_id="runwayml/stable-diffusion-inpainting",
        allow_patterns=["*.json", "*.txt", "*.fp16.safetensors", "*.model"],
        ignore_patterns=["*.bin", "*.msgpack", "*.h5", "*.ckpt"],
    )
    print("Stable Diffusion Inpainting fp16 đã sẵn sàng trong cache.")
except Exception as exc:
    raise RuntimeError(
        "Không tải được Stable Diffusion Inpainting. "
        "Hãy bật Internet trong Kaggle Settings rồi chạy lại."
    ) from exc

cmd = [
    sys.executable,
    "run_pipeline.py",
    "full",
    "--output_dir",
    "outputs/pcb_synthesis",
    "--camg_epochs",
    "10",
    "--camg_batch",
    "32",
    "--lora_epochs",
    "10",
    "--lora_batch",
    "32",
    "--num_samples",
    "400",
    "--yolo_epochs",
    "100",
    "--reject_ratio",
    "0.25",
    "--run_benchmark",
    "--benchmark_models",
    "yolo11s",
    "--benchmark_epochs",
    "100",
    "--benchmark_batch",
    "8",
    "--benchmark_conf",
    "0.25",
    "--benchmark_iou",
    "0.7",
    "--benchmark_imgsz",
    "600",
    "--benchmark_workers",
    "2",
    "--resume",
    "--device",
    "cuda",
]
print("Chạy:", " ".join(cmd))
subprocess.run(cmd, check=True)


---
# 🔬 PHƯƠNG ÁN B: CHẠY TỪNG BƯỚC THỦ CÔNG (THAM KHẢO)

Các ô bên dưới chỉ là **markdown tham khảo**, không tự chạy trong Save & Run All.
Nếu muốn chạy thủ công, copy lệnh vào một cell mới (thêm `!` trước `python`) hoặc
dùng terminal. Nếu đã chạy Phương án A thành công, không cần Phương án B.


### Step 1: Context-Aware Mask Generation (CAMG) — tham khảo

Sử dụng DINOv2 backbone để trích xuất đặc trưng vị trí ngữ nghĩa và Top-k Sampling
($k=5$) để sinh mask khuyết tật.

```bash
python run_pipeline.py step1_train \
    --metadata_path data/train_metadata.json \
    --camg_epochs 10 \
    --camg_batch 32 \
    --camg_lr 1e-4 \
    --device cuda

python run_pipeline.py step1 \
    --image_dir data/pcb_normal \
    --output_dir outputs/pcb_synthesis \
    --k 5
```


### Step 2a: Fine-tuning LoRA & Textual Inversion — tham khảo

```bash
python run_pipeline.py step2_train \
    --metadata_path data/train_metadata.json \
    --output_dir outputs/pcb_synthesis \
    --lora_epochs 10 \
    --lora_batch 32 \
    --lr_unet 1e-4 \
    --lr_text_encoder 5e-5 \
    --lr_ti 5e-4 \
    --device cuda
```


### Step 2b: Diffusion Inpainting Synthesis — tham khảo

```bash
python run_pipeline.py step2_generate \
    --mask_meta outputs/pcb_synthesis/masks/mask_metadata.json \
    --output_dir outputs/pcb_synthesis \
    --num_samples 400 \
    --inpaint_steps 50 \
    --guidance 7.5 \
    --device cuda
```


### Step 3: Confidence-Based Quality Filtering (YOLO11s) — tham khảo

```bash
python run_pipeline.py step3 \
    --gen_meta outputs/pcb_synthesis/generated/generated_metadata.json \
    --output_dir outputs/pcb_synthesis \
    --data_yaml data/id_setup.yaml \
    --yolo_size s \
    --yolo_epochs 100 \
    --reject_ratio 0.25 \
    --device cuda
```


### Step 4: Downstream Benchmark (chỉ YOLO11s) — tham khảo

```bash
python run_pipeline.py step4 \
    --models yolo11s \
    --split id \
    --base_yaml data/id_setup.yaml \
    --aug_yaml outputs/pcb_synthesis/filtered/augmented_dataset.yaml \
    --output_dir outputs/pcb_synthesis \
    --epochs 100 \
    --batch_size 8 \
    --lr0 1e-4 \
    --conf 0.25 \
    --iou 0.7 \
    --imgsz 600 \
    --workers 2 \
    --device cuda
```


---
## Bước 5: Xem Kết Quả & Trực Quan Hóa Báo Cáo

Xem trạng thái pipeline và báo cáo benchmark thực tế của YOLO11s. Báo cáo chỉ có sau khi Step 4 hoàn tất.


In [ ]:
# ── 5.1. Xem Dashboard Trạng Thái Pipeline ─────────────────
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "run_pipeline.py",
        "status",
        "--output_dir",
        "outputs/pcb_synthesis",
    ],
    check=True,
)

# ── 5.2. Hiển thị Báo Cáo Benchmark ─────────────────────────
from IPython.display import Markdown, display

report_file = Path("outputs/pcb_synthesis/benchmark/benchmark_report.md")
if report_file.exists():
    display(Markdown(report_file.read_text(encoding="utf-8")))
else:
    print("Chưa có benchmark_report.md. Hãy hoàn tất Step 4 để xem báo cáo thực tế.")


In [ ]:
# ── 5.3. Hiển thị mẫu ảnh khuyết tật tổng hợp đã qua lọc ──
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

accepted_imgs = list(Path("outputs/pcb_synthesis/filtered/accepted/images").glob("*.jpg"))
if not accepted_imgs:
    accepted_imgs = list(Path("outputs/pcb_synthesis/generated").glob("*.jpg"))

if accepted_imgs:
    samples = accepted_imgs[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_p in zip(axes.flat, samples):
        img = cv2.imread(str(img_p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_p.name, fontsize=10)
        ax.axis("off")
    plt.suptitle("Sample Synthetic Defect Images Generated by Inpainting Diffusion", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Chưa tìm thấy ảnh tổng hợp nào để hiển thị.")